In [1]:
import os
import glob
import cv2
import hashlib
import shutil

# Function to compute video hash (to remove duplicates)
def get_video_hash(video_path, num_frames=20, input_size=(224, 224)):
    cap = cv2.VideoCapture(video_path)
    frame_hashes = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_interval = max(1, total_frames // num_frames)
    count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if count % frame_interval == 0:
            frame = cv2.resize(frame, input_size)
            frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frame_bytes = frame_gray.tobytes()
            frame_hashes.append(hashlib.md5(frame_bytes).hexdigest())
        count += 1

    cap.release()
    return tuple(frame_hashes)

# Load dataset and remove duplicates
DATASET_PATH = "/kaggle/input/datashoplifting/Shop DataSet"
video_paths, labels = [], []
unique_video_hashes = {}

# Create output directories for unique videos
OUTPUT_SHOPLIFTERS = "/kaggle/working/Unique_Videos/shop_lifters"
OUTPUT_NON_SHOPLIFTERS = "/kaggle/working/Unique_Videos/non_shop_lifters"
os.makedirs(OUTPUT_SHOPLIFTERS, exist_ok=True)  # Create directory for shop lifters
os.makedirs(OUTPUT_NON_SHOPLIFTERS, exist_ok=True)  # Create directory for non-shop lifters

for label, category in enumerate(["non shop lifters", "shop lifters"]):
    video_folder = os.path.join(DATASET_PATH, category)
    video_files = glob.glob(os.path.join(video_folder, "*.mp4"))
    
    for video_file in video_files:
        video_hash = get_video_hash(video_file)
        if video_hash not in unique_video_hashes:
            unique_video_hashes[video_hash] = video_file
            video_paths.append(video_file)
            labels.append(label)

            # Copy unique video to the appropriate output directory
            if category == "shop lifters":
                shutil.copy(video_file, os.path.join(OUTPUT_SHOPLIFTERS, os.path.basename(video_file)))
            else:  # "non shop lifters"
                shutil.copy(video_file, os.path.join(OUTPUT_NON_SHOPLIFTERS, os.path.basename(video_file)))

print(f"Total Unique Videos: {len(video_paths)}")
print(f"Unique shop lifter videos have been saved to: {OUTPUT_SHOPLIFTERS}")
print(f"Unique non shop lifter videos have been saved to: {OUTPUT_NON_SHOPLIFTERS}")

Total Unique Videos: 637
Unique shop lifter videos have been saved to: /kaggle/working/Unique_Videos/shop_lifters
Unique non shop lifter videos have been saved to: /kaggle/working/Unique_Videos/non_shop_lifters


In [2]:
import cv2
import os
import glob

# Input and output paths
input_dirs = {
    'shop_lifters': '/kaggle/working/Unique_Videos/shop_lifters',
    'non_shop_lifters': '/kaggle/working/Unique_Videos/non_shop_lifters'
}
output_dir = '/kaggle/working/Shop_DataSet/frames'  # Use a writable directory
os.makedirs(output_dir, exist_ok=True)

frame_size = (224, 224)  # Resize size
frame_rate = 1  # Save one frame per second

for category, input_dir in input_dirs.items():
    video_files = glob.glob(os.path.join(input_dir, '*.mp4'))
    category_output_dir = os.path.join(output_dir, category)
    os.makedirs(category_output_dir, exist_ok=True)

    for video_file in video_files:
        cap = cv2.VideoCapture(video_file)
        video_name = os.path.splitext(os.path.basename(video_file))[0]

        # Create a subdirectory for each video
        video_output_dir = os.path.join(category_output_dir, video_name)
        os.makedirs(video_output_dir, exist_ok=True)

        frame_count = 0
        fps = int(cap.get(cv2.CAP_PROP_FPS))  # Get frames per second

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            # Save every nth frame based on frame_rate
            if frame_count % fps == 0:
                resized_frame = cv2.resize(frame, frame_size)
                frame_filename = os.path.join(video_output_dir, f'frame_{frame_count:04d}.jpg')
                cv2.imwrite(frame_filename, resized_frame)
            
            frame_count += 1
        
        cap.release()

print(f"Frames have been saved to: {output_dir}")

Frames have been saved to: /kaggle/working/Shop_DataSet/frames


In [3]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Define settings
input_dir = '/kaggle/working/Shop_DataSet/frames'
frame_size = (224, 224)
num_frames = 20  # Fixed number of frames

# Define categories
categories = ['shop_lifters', 'non_shop_lifters']
data = []
labels = []

# Step 1: Load frames, pad or truncate to num_frames
for category in categories:
    category_path = os.path.join(input_dir, category)
    label = categories.index(category)  # 0 or 1
    
    for video_folder in os.listdir(category_path):
        if video_folder.startswith('.'):  # Skip hidden files/folders
            continue
        
        video_path = os.path.join(category_path, video_folder)
        frames = []
        
        # Load frames in sorted order
        frame_files = sorted([f for f in os.listdir(video_path) if f.endswith('.jpg')])
        
        for frame_file in frame_files:
            frame_path = os.path.join(video_path, frame_file)
            frame = load_img(frame_path, target_size=frame_size)  # Resize to 128x128
            frame = img_to_array(frame)
            
            # ✅ Normalize to [0, 1]
            frame = frame / 255.0  
            
            frames.append(frame)

        # ✅ Pad or truncate to exactly `num_frames`
        if len(frames) < num_frames:
            padding = [np.zeros((224, 224, 3), dtype=np.float32)] * (num_frames - len(frames))
            frames.extend(padding)
        else:
            frames = frames[:num_frames]

        data.append(frames)
        labels.append(label)

# ✅ Convert to numpy arrays
data = np.array(data, dtype=np.float32)   
labels = np.array(labels, dtype=np.int32) 

# ✅ Print shapes to verify
print("Data shape:", data.shape)   
print("Labels shape:", labels.shape)  


Data shape: (637, 20, 224, 224, 3)
Labels shape: (637,)


In [4]:
import os
import numpy as np
from PIL import Image
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torchvision import models, transforms
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score

# Settings
input_dir = '/kaggle/working/Shop_DataSet/frames'
frame_size = (224, 224)
num_frames = 20
batch_size = 8
num_epochs = 15
learning_rate = 0.0001
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Define categories
categories = ['shop_lifters', 'non_shop_lifters']
data = []
labels = []

# Step 1: Load and preprocess data
transform = transforms.Compose([
    transforms.Resize(frame_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def process_frame(frame):
    frame = transform(frame)
    return frame

for category in categories:
    category_path = os.path.join(input_dir, category)
    label = categories.index(category)

    for video_folder in os.listdir(category_path):
        if video_folder.startswith('.'):
            continue

        video_path = os.path.join(category_path, video_folder)
        frames = []

        frame_files = sorted([f for f in os.listdir(video_path) if f.endswith('.jpg')])

        for frame_file in frame_files:
            frame_path = os.path.join(video_path, frame_file)
            frame = Image.open(frame_path).convert('RGB')
            frame = process_frame(frame)
            frames.append(frame)

        # ✅ Pad or truncate to `num_frames`
        if len(frames) < num_frames:
            padding = [torch.zeros((3, *frame_size))] * (num_frames - len(frames))
            frames.extend(padding)
        else:
            frames = frames[:num_frames]

        data.append(torch.stack(frames))
        labels.append(label)

# ✅ Convert to tensors
data = torch.stack(data)  # Shape: (num_samples, num_frames, 3, 224, 224)
labels = torch.tensor(labels).float()

# ✅ Split into train, validation, and test sets
train_data, temp_data, train_labels, temp_labels = train_test_split(
    data, labels, test_size=0.3, random_state=42, stratify=labels
)
val_data, test_data, val_labels, test_labels = train_test_split(
    temp_data, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

print(f"Train size: {train_data.shape[0]}, Validation size: {val_data.shape[0]}, Test size: {test_data.shape[0]}")

# Step 2: Define DataLoader
train_dataset = TensorDataset(train_data, train_labels)
val_dataset = TensorDataset(val_data, val_labels)
test_dataset = TensorDataset(test_data, test_labels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Step 3: Load EfficientNet Model
class VideoClassifier(nn.Module):
    def __init__(self, hidden_size=64):
        super(VideoClassifier, self).__init__()
        self.base_model = models.efficientnet_b0(pretrained=True)
        self.base_model = nn.Sequential(*list(self.base_model.children())[:-1])
        feature_dim = 1280

        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        batch_size, num_frames, C, H, W = x.shape
        x = x.view(batch_size * num_frames, C, H, W)

        with torch.no_grad():
            x = torch.flatten(self.base_model(x), start_dim=1)

        x = x.view(batch_size, num_frames, -1)
        x, _ = self.lstm(x)
        x = x[:, -1, :]  # Take last LSTM output
        x = self.fc(x)
        return x

# ✅ Initialize model
model = VideoClassifier().to(device)

# Step 4: Define optimizer and loss
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.BCEWithLogitsLoss()

# Step 5: Train the model
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    all_labels = []
    all_preds = []

    for inputs, labels in tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs).squeeze()
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        predicted = (torch.sigmoid(outputs) > 0.5).float()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    train_loss /= len(train_loader)
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

    # ✅ Validation
    model.eval()
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs).squeeze()
            loss = loss_fn(outputs, labels)
            val_loss += loss.item()

            predicted = (torch.sigmoid(outputs) > 0.5).float()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    val_loss /= len(val_loader)
    val_acc = (np.array(all_preds) == np.array(all_labels)).mean()
    print(f"Validation Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

# ✅ Step 6: Evaluate on test set
model.eval()
test_loss = 0
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs).squeeze()
        loss = loss_fn(outputs, labels)
        test_loss += loss.item()

        predicted = (torch.sigmoid(outputs) > 0.5).float()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

test_loss /= len(test_loader)
test_acc = (np.array(all_preds) == np.array(all_labels)).mean()
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")


Train size: 445, Validation size: 96, Test size: 96


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 122MB/s] 
Training Epoch 1/15: 100%|██████████| 56/56 [11:41<00:00, 12.53s/it]


Epoch 1 | Train Loss: 0.6951 | Precision: 0.4877 | Recall: 0.6347 | F1: 0.5516
Validation Loss: 0.7003 | Acc: 0.4896 | Precision: 0.4444 | Recall: 0.1702 | F1: 0.2462


Training Epoch 2/15: 100%|██████████| 56/56 [12:03<00:00, 12.92s/it]


Epoch 2 | Train Loss: 0.6871 | Precision: 0.6494 | Recall: 0.2283 | F1: 0.3378
Validation Loss: 0.6970 | Acc: 0.5208 | Precision: 0.5143 | Recall: 0.3830 | F1: 0.4390


...

Epoch 13 | Train Loss: 0.2878 | Precision: 0.9538 | Recall: 0.8493 | F1: 0.8986
Validation Loss: 0.2415 | Acc: 0.9479 | Precision: 0.9773 | Recall: 0.9149 | F1: 0.9451


Training Epoch 14/15: 100%|██████████| 56/56 [11:42<00:00, 12.54s/it]


Epoch 14 | Train Loss: 0.2738 | Precision: 0.9500 | Recall: 0.8676 | F1: 0.9069
Validation Loss: 0.2357 | Acc: 0.9479 | Precision: 0.9773 | Recall: 0.9149 | F1: 0.9451


Training Epoch 15/15: 100%|██████████| 56/56 [11:45<00:00, 12.59s/it]


Epoch 15 | Train Loss: 0.2280 | Precision: 0.9801 | Recall: 0.8995 | F1: 0.9381
Validation Loss: 0.2189 | Acc: 0.9479 | Precision: 0.9773 | Recall: 0.9149 | F1: 0.9451
Test Loss: 0.1241 | Test Acc: 0.9792 | Precision: 1.0000 | Recall: 0.9574 | F1: 0.9783
